# Installation

In [ ]:
#pip uninstall -y transformers torch torchvision

In [ ]:
#!pip install git+https://github.com/dnth/rag-datakit.git

In [12]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import BatchSamplers
from datasets import load_dataset

In [13]:
# dataset = load_dataset("frankwong2001/ssf-train-valid-full-synthetic-batch10")
# dataset = load_dataset("frankwong2001/ssf-train-valid-full-synthetic-batch10-cleaned")
dataset = load_dataset("frankwong2001/ssf-train-valid-full-synthetic-v2")
dataset

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 3016
    })
    valid: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 754
    })
})

In [14]:
dataset['valid'][0]

{'anchor': 'The Director works in the field of social work management. He/She should have expertise and experience in domains under social work management in to supervise strategic initiatives, corporate governance, resource management, organisation and capability development. He develops and reviews framework for the organisations operating guidelines and standards, directs the implementation of corporate policies in accordance with governance regulations and drives improvements to the service delivery and operational efficiency. He is responsible for developing resource allocation and human resource management systems as well as fostering collaborations with external agencies. A highly experienced management staff who possesses excellent management and leadership skills, the Director works in institutional settings, communities, Voluntary Welfare Organisations and hospitals. He also works in collaboration with other agencies and ministries in the course of his work.',
 'positive': 'T

# W&B and Model Configuration

In [15]:
import wandb
import os
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Fetch the WANDB_API_KEY from the environment
wandb_api_key = os.getenv("WAB_API_KEY")

# Log in using the API key
wandb.login(key=wandb_api_key)

model_id = "Qwen/Qwen3-Embedding-0.6B"
save_model_path = "./models/Qwen/Qwen3-Embedding-0.6B"
wandb.init(project="rag-datakit-finetunes", name="Attempt_6_Qwen3-Embedding-0.6B~frankwong2001/ssf-train-valid-full-synthetic-v2")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


# Training Arguments

In [16]:
args = SentenceTransformerTrainingArguments(
    output_dir=save_model_path,
    num_train_epochs=5,                         # number of epochs
    per_device_train_batch_size=32,             # train batch size
    gradient_accumulation_steps=16,             # for a global batch size of 512
    per_device_eval_batch_size=16,              # evaluation batch size
    warmup_ratio=0.1,                           # warmup ratio
    learning_rate=2e-5,                         # learning rate, 2e-5 is a good value
    lr_scheduler_type="cosine",                 # use cosine learning rate scheduler
    optim="adamw_torch_fused",
    tf32=False,                                 # use tf32 precision
    bf16=True,         
    #fp16=True,                                                  # use bf16 precision
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
    eval_strategy="epoch",                      # evaluate after each epoch
    save_strategy="epoch",                      # save after each epoch
    logging_strategy="epoch",                   # log after each epoch
    save_total_limit=3,                         # save only the last 3 models
    load_best_model_at_end=True,                # load the best model when training ends
    report_to="wandb",
    gradient_checkpointing=True,              # use fused adamw optimizer
    #use_cache=False                            # disable the use of cache
    weight_decay=0.01,                             # apply weight decay
    max_grad_norm=0.5,                           # clip the gradient norm
    warmup_steps=1500                           # number of warmup steps
    )

In [17]:
model = SentenceTransformer(model_id)
train_loss = MultipleNegativesRankingLoss(model)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['valid'],  
    loss=train_loss,
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

# Execute Training


In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.055600,0.042626
2,0.056200,0.042341
3,0.052300,0.039281
4,0.049100,0.037919
5,0.040200,0.029482


TrainOutput(global_step=30, training_loss=0.050678070882956186, metrics={'train_runtime': 930.6009, 'train_samples_per_second': 16.205, 'train_steps_per_second': 0.032, 'total_flos': 0.0, 'train_loss': 0.050678070882956186, 'epoch': 5.0})

#  Save & Upload Model

In [19]:
trainer.save_model()

In [20]:
import os
wandb.save(os.path.join(save_model_path, "*"))

wandb: WARNING Symlinked 19 files into the W&B run directory, call wandb.save again to sync new files.


['/root/rag-datakit/nbs-frank/wandb/run-20250910_084702-0el46pqh/files/models/Qwen/Qwen3-Embedding-0.6B/checkpoint-30',
 '/root/rag-datakit/nbs-frank/wandb/run-20250910_084702-0el46pqh/files/models/Qwen/Qwen3-Embedding-0.6B/added_tokens.json',
 '/root/rag-datakit/nbs-frank/wandb/run-20250910_084702-0el46pqh/files/models/Qwen/Qwen3-Embedding-0.6B/tokenizer_config.json',
 '/root/rag-datakit/nbs-frank/wandb/run-20250910_084702-0el46pqh/files/models/Qwen/Qwen3-Embedding-0.6B/1_Pooling',
 '/root/rag-datakit/nbs-frank/wandb/run-20250910_084702-0el46pqh/files/models/Qwen/Qwen3-Embedding-0.6B/model.safetensors',
 '/root/rag-datakit/nbs-frank/wandb/run-20250910_084702-0el46pqh/files/models/Qwen/Qwen3-Embedding-0.6B/modules.json',
 '/root/rag-datakit/nbs-frank/wandb/run-20250910_084702-0el46pqh/files/models/Qwen/Qwen3-Embedding-0.6B/chat_template.jinja',
 '/root/rag-datakit/nbs-frank/wandb/run-20250910_084702-0el46pqh/files/models/Qwen/Qwen3-Embedding-0.6B/vocab.json',
 '/root/rag-datakit/nbs-fr

In [21]:
wandb.finish()

eval/loss,██▆▅▁
eval/runtime,▂▁▄▄█
eval/samples_per_second,▇█▅▅▁
eval/steps_per_second,▇█▅▅▁
train/epoch,▁▁▃▃▅▅▆▆███
train/global_step,▁▁▃▃▅▅▆▆███
train/grad_norm,▆█▆▃▁
train/learning_rate,▁▃▅▆█
train/loss,██▆▅▁
eval/loss,0.02948
eval/runtime,9.2675


# Push to Hugging Face

In [22]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import Trainer

# Load environment variables from .env file
load_dotenv()

# Fetch the Hugging Face API key from the environment
hf_api_key = os.getenv("HF_TOKEN")

# Log in using the Hugging Face API key
login(token=hf_api_key)

# Assuming you have a Trainer object `trainer`
trainer.model.push_to_hub("frankwong2001/6_attempt_Qwen3-Embedding-0.6B", exist_ok=True)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmpnmrexz00/tokenizer.json       : 100%|##########| 11.4MB / 11.4MB            

  /tmp/tmpnmrexz00/model.safetensors    :   0%|          | 1.19MB / 2.38GB            

'https://huggingface.co/frankwong2001/6_attempt_Qwen3-Embedding-0.6B/commit/a5b8dae93751c534fad39d31bd34b7f382f2101c'